# System B — ColPali v1.3 on ViDoRe DocVQA (Colab)

Thin wrapper around `scripts/run_colpali.py` with `configs/colpali_v1.3_docvqa.yaml`: encodes all 500 pages and 451 queries, computes the full score matrix with the official protocol, evaluates it, and measures resources.

**Before running:** `Runtime → Change runtime type → L4 GPU` (native bf16; peak memory in the smoke test was 16.2 GB, too close to a T4's 15 GB).

Outputs are zipped at the end and downloaded so they can be committed to `results/colpali_v1.3_docvqa/`.

In [ ]:
!nvidia-smi

In [ ]:
# Idempotent: clones on first run, pulls the latest commit when the runtime is reused.
import os
if not os.path.isdir("/content/colpali-hybrid-retrieval"):
    !git clone https://github.com/ismailucrn/colpali-hybrid-retrieval.git /content/colpali-hybrid-retrieval
%cd /content/colpali-hybrid-retrieval
!git pull -q
!git log -1 --format='%H %s'

In [ ]:
# Colab preinstalls torchao 0.10, which peft 0.20 rejects at import time; this project does not use it.
!pip uninstall -y -q torchao
!pip install -q -r requirements.txt

In [ ]:
!python -m pytest -q tests

In [ ]:
!python scripts/run_colpali.py --config configs/colpali_v1.3_docvqa.yaml

In [ ]:
!mkdir -p results/colpali_v1.3_docvqa
!pip freeze > results/colpali_v1.3_docvqa/pip_freeze_colab.txt
!ls -la results/colpali_v1.3_docvqa
!cat results/colpali_v1.3_docvqa/metrics.json

In [ ]:
# Zip the outputs and download them (scores.npz is binary, so it cannot be copied from the page).
!cd results && zip -q -r /content/colpali_v1.3_docvqa.zip colpali_v1.3_docvqa
from google.colab import files
files.download("/content/colpali_v1.3_docvqa.zip")